# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_06 — Microstructure Friction C++ Core

### Research question

How can we build a fast, auditable execution core that applies realistic microstructure frictions — bid/ask crossing, tick rounding, latency, partial fills, fees, queue constraints, and price-limit checks — while validating Python and C++ implementations against each other?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
06_06_microstructure_friction_cpp_core
```

The previous notebooks introduced optimal execution, dynamic execution control, market impact, Hawkes order flow, and rough volatility. This notebook turns those ideas into a low-level execution simulation core.

It covers:

1. L1 top-of-book quote stream;
2. bid/ask spread;
3. tick-size rounding;
4. market order fills;
5. limit order fill rules;
6. latency;
7. partial fills;
8. visible depth constraints;
9. queue-position proxy;
10. taker fees and maker rebates;
11. price-limit checks;
12. realised slippage;
13. implementation shortfall;
14. Python reference engine;
15. C++ shared-library core;
16. `ctypes` wrapper;
17. Python-vs-C++ parity tests;
18. benchmark timing;
19. fill-quality diagnostics;
20. governance flags;
21. saved outputs and manifest.

The key idea:

> Microstructure friction is where a backtest stops being a return spreadsheet and becomes an execution system.

## 1. Why a C++ execution core?

Python is excellent for research, diagnostics, plotting, and notebooks.

But execution simulation loops can be slow when they process:

- millions of quotes;
- thousands of orders;
- latency-adjusted timestamps;
- partial fills;
- tick rounding;
- queue constraints;
- event-by-event accounting.

A C++ core is useful for:

1. speed;
2. deterministic behaviour;
3. explicit memory layout;
4. easier production translation;
5. parity testing against a Python reference.

The correct workflow is:

```text
Python reference -> C++ kernel -> parity tests -> benchmark -> governance
```

Never trust the C++ version until it matches the Python version on controlled test cases.

## 2. Microstructure friction model

This notebook models L1 top-of-book data only:

$$
bid_t,\ ask_t,\ bidSize_t,\ askSize_t
$$

No full depth.

No hidden liquidity.

No true queue reconstruction.

But even L1 is enough to model important frictions:

### Market buy

Filled at ask, subject to available ask size.

### Market sell

Filled at bid, subject to available bid size.

### Limit buy

Filled when limit price crosses or equals ask.

### Limit sell

Filled when limit price crosses or equals bid.

### Latency

Order submitted at event $t$ is evaluated at:

$$
t + latency
$$

### Tick rounding

Buy limit prices are rounded down to tick.

Sell limit prices are rounded up to tick.

### Fees

Taker fees apply to marketable fills.

Maker rebates can apply to passive fills.

## 3. Core assumptions

This is an educational execution core, not a full exchange simulator.

Assumptions:

1. L1 visible size limits maximum fill per event.
2. Queue position is approximated by a fill fraction.
3. Marketable limit orders are treated as taker orders.
4. Passive limit orders fill only when price condition is met.
5. Price limits block orders outside allowed bands.
6. No order persistence beyond a simple expiry horizon.
7. No cancel-replace queue penalty beyond latency.
8. No matching-engine priority rules beyond the queue proxy.

The point is not to pretend this is a real exchange.

The point is to create a fast, explicit, testable friction layer.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import ctypes
import json
import os
import platform
import shutil
import subprocess
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class MicrostructureCppConfig:
    n_events: int = 80_000
    n_orders: int = 6_000
    seed: int = 42
    initial_mid: float = 100.0
    tick_size: float = 0.01
    base_spread_ticks: int = 1
    max_spread_ticks: int = 8
    base_depth: float = 2_000.0
    latency_events: int = 3
    passive_fill_fraction: float = 0.35
    taker_fee_bps: float = 0.45
    maker_rebate_bps: float = -0.05
    price_limit_pct: float = 0.10
    order_expiry_events: int = 25
    benchmark_repeats: int = 5
    output_subdir: str = "microstructure_friction_cpp_core_v1"

config = MicrostructureCppConfig()
config

## 5. Simulate L1 top-of-book quote stream

We simulate:

- mid price;
- bid/ask prices;
- bid/ask sizes;
- spread;
- price-limit bands.

The stream is event-indexed rather than timestamped for simplicity.

In [ ]:
def round_to_tick(x, tick):
    return np.round(x / tick) * tick

def simulate_l1_quotes(config: MicrostructureCppConfig):
    rng = np.random.default_rng(config.seed)
    n = config.n_events

    event_id = np.arange(n, dtype=np.int64)

    returns = rng.standard_t(df=6, size=n) * 0.00025
    shock_idx = rng.choice(n, size=max(5, n // 5000), replace=False)
    returns[shock_idx] += rng.normal(0.0, 0.004, size=len(shock_idx))

    mid = config.initial_mid * np.exp(np.cumsum(returns))
    mid = round_to_tick(mid, config.tick_size)

    spread_ticks = rng.choice(
        np.arange(1, config.max_spread_ticks + 1),
        size=n,
        p=np.array([0.50, 0.22, 0.11, 0.06, 0.04, 0.03, 0.025, 0.015]),
    )
    spread = spread_ticks * config.tick_size

    bid = round_to_tick(mid - spread / 2, config.tick_size)
    ask = round_to_tick(mid + spread / 2, config.tick_size)
    ask = np.maximum(ask, bid + config.tick_size)

    bid_size = rng.lognormal(np.log(config.base_depth), 0.55, size=n)
    ask_size = rng.lognormal(np.log(config.base_depth), 0.55, size=n)

    # Stress episodes: wider spreads and thinner depth.
    stress = np.zeros(n, dtype=int)
    for start in rng.choice(np.arange(100, n - 500), size=8, replace=False):
        length = int(rng.integers(80, 300))
        end = min(n, start + length)
        stress[start:end] = 1
        bid_size[start:end] *= rng.uniform(0.20, 0.55)
        ask_size[start:end] *= rng.uniform(0.20, 0.55)
        extra = rng.integers(1, 5, size=end - start) * config.tick_size
        bid[start:end] -= extra
        ask[start:end] += extra

    limit_down = config.initial_mid * (1.0 - config.price_limit_pct)
    limit_up = config.initial_mid * (1.0 + config.price_limit_pct)

    quotes = pd.DataFrame({
        "event_id": event_id,
        "bid": bid,
        "ask": ask,
        "mid": (bid + ask) / 2,
        "bid_size": bid_size,
        "ask_size": ask_size,
        "spread": ask - bid,
        "spread_ticks": (ask - bid) / config.tick_size,
        "stress": stress,
        "limit_down": limit_down,
        "limit_up": limit_up,
    })

    return quotes

quotes = simulate_l1_quotes(config)

quotes.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(quotes["event_id"], quotes["mid"])
plt.title("Simulated L1 Mid Price")
plt.xlabel("Event")
plt.ylabel("Mid")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(quotes["event_id"], quotes["spread_ticks"])
plt.title("Spread in Ticks")
plt.xlabel("Event")
plt.ylabel("Spread ticks")
plt.show()

## 6. Generate synthetic parent/child order stream

Order fields:

- `submit_event`;
- `side`: +1 buy, -1 sell;
- `order_type`: 0 market, 1 limit;
- `quantity`;
- `limit_price`;
- `arrival_mid`;
- `expiry_event`.

The order generator creates both market and limit orders.

Limit prices are placed around the bid/ask, so some are passive and some are marketable.

In [ ]:
def generate_orders(quotes: pd.DataFrame, config: MicrostructureCppConfig):
    rng = np.random.default_rng(config.seed + 101)
    n_orders = config.n_orders

    submit_event = np.sort(rng.choice(np.arange(0, config.n_events - config.order_expiry_events - config.latency_events - 2), size=n_orders, replace=False))
    side = rng.choice(np.array([1, -1], dtype=np.int32), size=n_orders)
    order_type = rng.choice(np.array([0, 1], dtype=np.int32), size=n_orders, p=[0.35, 0.65])

    q = rng.lognormal(np.log(800), 0.80, size=n_orders)
    q = np.maximum(1.0, np.round(q))

    limit_price = np.zeros(n_orders, dtype=float)
    arrival_mid = quotes.loc[submit_event, "mid"].to_numpy()

    for i, ev in enumerate(submit_event):
        bid = quotes.loc[ev, "bid"]
        ask = quotes.loc[ev, "ask"]

        if order_type[i] == 0:
            limit_price[i] = 0.0
            continue

        # Limit orders: mixture of passive and marketable.
        if side[i] > 0:
            passive_price = bid
            marketable_price = ask + rng.integers(0, 2) * config.tick_size
            limit_price[i] = passive_price if rng.uniform() < 0.70 else marketable_price
            limit_price[i] = np.floor(limit_price[i] / config.tick_size) * config.tick_size
        else:
            passive_price = ask
            marketable_price = bid - rng.integers(0, 2) * config.tick_size
            limit_price[i] = passive_price if rng.uniform() < 0.70 else marketable_price
            limit_price[i] = np.ceil(limit_price[i] / config.tick_size) * config.tick_size

    expiry_event = submit_event + config.order_expiry_events

    orders = pd.DataFrame({
        "order_id": np.arange(n_orders, dtype=np.int64),
        "submit_event": submit_event.astype(np.int64),
        "side": side.astype(np.int32),
        "side_name": np.where(side > 0, "BUY", "SELL"),
        "order_type": order_type.astype(np.int32),
        "order_type_name": np.where(order_type == 0, "MARKET", "LIMIT"),
        "quantity": q.astype(float),
        "limit_price": limit_price,
        "arrival_mid": arrival_mid,
        "expiry_event": expiry_event.astype(np.int64),
    })

    return orders

orders = generate_orders(quotes, config)

orders.head(), orders["order_type_name"].value_counts()

## 7. Python reference execution engine

This reference engine prioritises clarity over speed.

Rules:

### Market order

- Buy fills at ask.
- Sell fills at bid.
- Fill is capped by visible L1 size.
- Taker fee applies.

### Limit order

After latency, scan until expiry.

- Buy limit fills if limit price >= ask.
- Sell limit fills if limit price <= bid.
- Marketable limit fills are treated as taker.
- Passive fill uses a queue proxy: only a fraction of displayed size is available.
- Maker rebate applies to passive fills.

### Price limits

Orders outside price-limit bands are rejected.

In [ ]:
def execute_orders_python(quotes, orders, config):
    fill_rows = []

    bid = quotes["bid"].to_numpy()
    ask = quotes["ask"].to_numpy()
    bid_size = quotes["bid_size"].to_numpy()
    ask_size = quotes["ask_size"].to_numpy()
    mid = quotes["mid"].to_numpy()
    limit_down = quotes["limit_down"].to_numpy()
    limit_up = quotes["limit_up"].to_numpy()

    for _, order in orders.iterrows():
        order_id = int(order["order_id"])
        side = int(order["side"])
        order_type = int(order["order_type"])
        qty = float(order["quantity"])
        limit_price = float(order["limit_price"])
        arrival_mid = float(order["arrival_mid"])

        start = int(order["submit_event"]) + config.latency_events
        end = min(int(order["expiry_event"]), len(quotes) - 1)

        remaining = qty
        filled = 0.0
        notional = 0.0
        fees = 0.0
        first_fill_event = -1
        reject_reason = "NONE"

        # Price-limit validity for limit orders.
        if order_type == 1:
            if limit_price < limit_down[start] or limit_price > limit_up[start]:
                reject_reason = "PRICE_LIMIT_REJECT"
                fill_rows.append({
                    "order_id": order_id,
                    "filled_qty": 0.0,
                    "avg_fill_price": np.nan,
                    "notional": 0.0,
                    "fees": 0.0,
                    "first_fill_event": -1,
                    "last_fill_event": -1,
                    "remaining_qty": remaining,
                    "reject_reason": reject_reason,
                    "is_complete": False,
                    "implementation_shortfall_bps": np.nan,
                })
                continue

        last_fill_event = -1

        for ev in range(start, end + 1):
            if remaining <= 1e-12:
                break

            if side > 0:
                executable_price = ask[ev]
                visible_size = ask_size[ev]
                price_condition = True if order_type == 0 else limit_price >= ask[ev]
            else:
                executable_price = bid[ev]
                visible_size = bid_size[ev]
                price_condition = True if order_type == 0 else limit_price <= bid[ev]

            if not price_condition:
                continue

            marketable_limit = order_type == 1 and (
                (side > 0 and limit_price >= ask[ev])
                or (side < 0 and limit_price <= bid[ev])
            )

            is_taker = order_type == 0 or marketable_limit

            available = visible_size if is_taker else visible_size * config.passive_fill_fraction
            fill_qty = min(remaining, max(0.0, available))

            if fill_qty <= 0:
                continue

            fee_bps = config.taker_fee_bps if is_taker else config.maker_rebate_bps
            fee = fill_qty * executable_price * fee_bps / 10000.0

            filled += fill_qty
            remaining -= fill_qty
            notional += fill_qty * executable_price
            fees += fee

            if first_fill_event < 0:
                first_fill_event = ev
            last_fill_event = ev

        if filled > 0:
            avg_price = notional / filled
            if side > 0:
                is_bps = (avg_price - arrival_mid) / arrival_mid * 10000.0
            else:
                is_bps = (arrival_mid - avg_price) / arrival_mid * 10000.0
            is_bps += fees / max(notional, 1e-12) * 10000.0
        else:
            avg_price = np.nan
            is_bps = np.nan
            reject_reason = "NO_FILL" if reject_reason == "NONE" else reject_reason

        fill_rows.append({
            "order_id": order_id,
            "filled_qty": filled,
            "avg_fill_price": avg_price,
            "notional": notional,
            "fees": fees,
            "first_fill_event": first_fill_event,
            "last_fill_event": last_fill_event,
            "remaining_qty": remaining,
            "reject_reason": reject_reason,
            "is_complete": remaining <= 1e-12,
            "implementation_shortfall_bps": is_bps,
        })

    return pd.DataFrame(fill_rows)

fills_python = execute_orders_python(quotes, orders, config)

fills_python.head()

## 8. Python fill diagnostics

In [ ]:
def fill_quality_report(fills, orders):
    merged = fills.merge(orders[["order_id", "side_name", "order_type_name", "quantity"]], on="order_id", how="left")
    merged["fill_rate"] = merged["filled_qty"] / merged["quantity"]

    summary = pd.DataFrame([{
        "n_orders": len(merged),
        "fill_rate_mean": merged["fill_rate"].mean(),
        "completion_rate": merged["is_complete"].mean(),
        "no_fill_rate": (merged["filled_qty"] <= 0).mean(),
        "mean_shortfall_bps": merged["implementation_shortfall_bps"].mean(),
        "p95_shortfall_bps": merged["implementation_shortfall_bps"].quantile(0.95),
        "total_notional": merged["notional"].sum(),
        "total_fees": merged["fees"].sum(),
    }])

    by_type = (
        merged
        .groupby(["order_type_name", "side_name"])
        .agg(
            n_orders=("order_id", "count"),
            fill_rate_mean=("fill_rate", "mean"),
            completion_rate=("is_complete", "mean"),
            mean_shortfall_bps=("implementation_shortfall_bps", "mean"),
            p95_shortfall_bps=("implementation_shortfall_bps", lambda x: x.quantile(0.95)),
        )
        .reset_index()
    )

    return merged, summary, by_type

fills_python_merged, python_fill_summary, python_fill_by_type = fill_quality_report(fills_python, orders)

python_fill_summary, python_fill_by_type

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(fills_python_merged["implementation_shortfall_bps"].dropna(), bins=80)
plt.title("Python Engine Implementation Shortfall")
plt.xlabel("Shortfall, bps")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(fills_python_merged["fill_rate"], bins=50)
plt.title("Fill Rate Distribution")
plt.xlabel("Filled quantity / order quantity")
plt.ylabel("Count")
plt.show()

## 9. C++ core source

The C++ core processes arrays only.

This makes the memory contract explicit:

### Quote arrays

- bid
- ask
- bid size
- ask size
- mid
- price limits

### Order arrays

- submit event
- expiry event
- side
- type
- quantity
- limit price
- arrival mid

### Output arrays

- filled quantity
- average fill price
- notional
- fees
- first fill event
- last fill event
- remaining quantity
- reject code
- completion flag
- implementation shortfall

The goal is parity with the Python reference.

In [ ]:
cpp_source = r"""
#include <cmath>
#include <cstdint>
#include <algorithm>

extern "C" {

void execute_l1_orders(
    const int n_quotes,
    const int n_orders,
    const double* bid,
    const double* ask,
    const double* bid_size,
    const double* ask_size,
    const double* mid,
    const double* limit_down,
    const double* limit_up,
    const int64_t* submit_event,
    const int64_t* expiry_event,
    const int* side,
    const int* order_type,
    const double* quantity,
    const double* limit_price,
    const double* arrival_mid,
    const int latency_events,
    const double passive_fill_fraction,
    const double taker_fee_bps,
    const double maker_rebate_bps,
    double* out_filled_qty,
    double* out_avg_fill_price,
    double* out_notional,
    double* out_fees,
    int64_t* out_first_fill_event,
    int64_t* out_last_fill_event,
    double* out_remaining_qty,
    int* out_reject_code,
    int* out_is_complete,
    double* out_shortfall_bps
) {
    for (int i = 0; i < n_orders; ++i) {
        int64_t start = submit_event[i] + latency_events;
        int64_t end = expiry_event[i];

        if (start < 0) start = 0;
        if (end >= n_quotes) end = n_quotes - 1;
        if (start >= n_quotes) start = n_quotes - 1;

        const int s = side[i];
        const int typ = order_type[i];
        const double qty = quantity[i];
        const double lim = limit_price[i];
        const double arr_mid = arrival_mid[i];

        double remaining = qty;
        double filled = 0.0;
        double notional = 0.0;
        double fees = 0.0;
        int64_t first_fill = -1;
        int64_t last_fill = -1;
        int reject_code = 0;

        if (typ == 1) {
            if (lim < limit_down[start] || lim > limit_up[start]) {
                reject_code = 2;
                out_filled_qty[i] = 0.0;
                out_avg_fill_price[i] = NAN;
                out_notional[i] = 0.0;
                out_fees[i] = 0.0;
                out_first_fill_event[i] = -1;
                out_last_fill_event[i] = -1;
                out_remaining_qty[i] = remaining;
                out_reject_code[i] = reject_code;
                out_is_complete[i] = 0;
                out_shortfall_bps[i] = NAN;
                continue;
            }
        }

        for (int64_t ev = start; ev <= end; ++ev) {
            if (remaining <= 1e-12) break;

            double executable_price;
            double visible_size;
            bool price_condition;

            if (s > 0) {
                executable_price = ask[ev];
                visible_size = ask_size[ev];
                price_condition = (typ == 0) ? true : (lim >= ask[ev]);
            } else {
                executable_price = bid[ev];
                visible_size = bid_size[ev];
                price_condition = (typ == 0) ? true : (lim <= bid[ev]);
            }

            if (!price_condition) continue;

            bool marketable_limit = false;
            if (typ == 1) {
                if (s > 0 && lim >= ask[ev]) marketable_limit = true;
                if (s < 0 && lim <= bid[ev]) marketable_limit = true;
            }

            bool is_taker = (typ == 0) || marketable_limit;

            double available = is_taker ? visible_size : visible_size * passive_fill_fraction;
            if (available < 0.0) available = 0.0;

            double fill_qty = std::min(remaining, available);
            if (fill_qty <= 0.0) continue;

            double fee_bps = is_taker ? taker_fee_bps : maker_rebate_bps;
            double fee = fill_qty * executable_price * fee_bps / 10000.0;

            filled += fill_qty;
            remaining -= fill_qty;
            notional += fill_qty * executable_price;
            fees += fee;

            if (first_fill < 0) first_fill = ev;
            last_fill = ev;
        }

        double avg_price = NAN;
        double is_bps = NAN;

        if (filled > 0.0) {
            avg_price = notional / filled;
            if (s > 0) {
                is_bps = (avg_price - arr_mid) / arr_mid * 10000.0;
            } else {
                is_bps = (arr_mid - avg_price) / arr_mid * 10000.0;
            }
            if (notional > 1e-12) {
                is_bps += fees / notional * 10000.0;
            }
        } else {
            reject_code = (reject_code == 0) ? 1 : reject_code;
        }

        out_filled_qty[i] = filled;
        out_avg_fill_price[i] = avg_price;
        out_notional[i] = notional;
        out_fees[i] = fees;
        out_first_fill_event[i] = first_fill;
        out_last_fill_event[i] = last_fill;
        out_remaining_qty[i] = remaining;
        out_reject_code[i] = reject_code;
        out_is_complete[i] = (remaining <= 1e-12) ? 1 : 0;
        out_shortfall_bps[i] = is_bps;
    }
}

}
"""

output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

cpp_path = output_dir / "microstructure_friction_core.cpp"
cpp_path.write_text(cpp_source, encoding="utf-8")

cpp_path

## 10. Compile C++ core

Compilation is optional.

If `g++` is unavailable, the notebook falls back to Python-only mode.

This keeps the notebook portable.

In [ ]:
def compile_cpp_core(cpp_path: Path):
    compiler = shutil.which("g++") or shutil.which("clang++")
    if compiler is None:
        return None, "No C++ compiler found."

    system = platform.system().lower()
    if "windows" in system:
        lib_path = cpp_path.with_suffix(".dll")
        cmd = [compiler, "-O3", "-std=c++17", "-shared", "-o", str(lib_path), str(cpp_path)]
    elif "darwin" in system:
        lib_path = cpp_path.with_suffix(".dylib")
        cmd = [compiler, "-O3", "-std=c++17", "-shared", "-fPIC", "-o", str(lib_path), str(cpp_path)]
    else:
        lib_path = cpp_path.with_suffix(".so")
        cmd = [compiler, "-O3", "-std=c++17", "-shared", "-fPIC", "-o", str(lib_path), str(cpp_path)]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        return None, result.stderr

    return lib_path, "compiled"

lib_path, compile_message = compile_cpp_core(cpp_path)

lib_path, compile_message

## 11. C++ `ctypes` wrapper

The wrapper:

1. converts pandas columns to contiguous NumPy arrays;
2. allocates output arrays;
3. calls C++ core;
4. converts results back to DataFrame.

In [ ]:
def execute_orders_cpp(quotes, orders, config, lib_path):
    if lib_path is None:
        raise RuntimeError("C++ library unavailable.")

    lib = ctypes.CDLL(str(lib_path))

    func = lib.execute_l1_orders
    func.argtypes = [
        ctypes.c_int,
        ctypes.c_int,
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.int64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.int64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.int32, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.int32, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        ctypes.c_int,
        ctypes.c_double,
        ctypes.c_double,
        ctypes.c_double,
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.int64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.int64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.int32, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.int32, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
    ]
    func.restype = None

    n_quotes = len(quotes)
    n_orders = len(orders)

    def arr_float(col):
        return np.ascontiguousarray(col.to_numpy(dtype=np.float64))

    def arr_int64(col):
        return np.ascontiguousarray(col.to_numpy(dtype=np.int64))

    def arr_int32(col):
        return np.ascontiguousarray(col.to_numpy(dtype=np.int32))

    out_filled_qty = np.zeros(n_orders, dtype=np.float64)
    out_avg_fill_price = np.zeros(n_orders, dtype=np.float64)
    out_notional = np.zeros(n_orders, dtype=np.float64)
    out_fees = np.zeros(n_orders, dtype=np.float64)
    out_first_fill_event = np.zeros(n_orders, dtype=np.int64)
    out_last_fill_event = np.zeros(n_orders, dtype=np.int64)
    out_remaining_qty = np.zeros(n_orders, dtype=np.float64)
    out_reject_code = np.zeros(n_orders, dtype=np.int32)
    out_is_complete = np.zeros(n_orders, dtype=np.int32)
    out_shortfall_bps = np.zeros(n_orders, dtype=np.float64)

    func(
        n_quotes,
        n_orders,
        arr_float(quotes["bid"]),
        arr_float(quotes["ask"]),
        arr_float(quotes["bid_size"]),
        arr_float(quotes["ask_size"]),
        arr_float(quotes["mid"]),
        arr_float(quotes["limit_down"]),
        arr_float(quotes["limit_up"]),
        arr_int64(orders["submit_event"]),
        arr_int64(orders["expiry_event"]),
        arr_int32(orders["side"]),
        arr_int32(orders["order_type"]),
        arr_float(orders["quantity"]),
        arr_float(orders["limit_price"]),
        arr_float(orders["arrival_mid"]),
        int(config.latency_events),
        float(config.passive_fill_fraction),
        float(config.taker_fee_bps),
        float(config.maker_rebate_bps),
        out_filled_qty,
        out_avg_fill_price,
        out_notional,
        out_fees,
        out_first_fill_event,
        out_last_fill_event,
        out_remaining_qty,
        out_reject_code,
        out_is_complete,
        out_shortfall_bps,
    )

    reject_map = {0: "NONE", 1: "NO_FILL", 2: "PRICE_LIMIT_REJECT"}

    return pd.DataFrame({
        "order_id": orders["order_id"].to_numpy(dtype=np.int64),
        "filled_qty": out_filled_qty,
        "avg_fill_price": out_avg_fill_price,
        "notional": out_notional,
        "fees": out_fees,
        "first_fill_event": out_first_fill_event,
        "last_fill_event": out_last_fill_event,
        "remaining_qty": out_remaining_qty,
        "reject_code": out_reject_code,
        "reject_reason": [reject_map.get(int(x), "UNKNOWN") for x in out_reject_code],
        "is_complete": out_is_complete.astype(bool),
        "implementation_shortfall_bps": out_shortfall_bps,
    })

if lib_path is not None:
    fills_cpp = execute_orders_cpp(quotes, orders, config, lib_path)
else:
    fills_cpp = None

fills_cpp.head() if fills_cpp is not None else compile_message

## 12. Python-vs-C++ parity tests

The C++ core should match Python reference results within tight tolerances.

We check:

- filled quantity;
- average fill price;
- notional;
- fees;
- first/last fill events;
- remaining quantity;
- reject reason;
- completion flag;
- implementation shortfall.

In [ ]:
def compare_python_cpp(fills_python, fills_cpp):
    if fills_cpp is None:
        return pd.DataFrame([{
            "metric": "cpp_available",
            "max_abs_diff": np.nan,
            "passed": False,
            "note": "C++ library unavailable; Python fallback only.",
        }])

    merged = fills_python.merge(
        fills_cpp,
        on="order_id",
        suffixes=("_py", "_cpp"),
        how="inner",
    )

    rows = []
    numeric_cols = [
        "filled_qty",
        "avg_fill_price",
        "notional",
        "fees",
        "remaining_qty",
        "implementation_shortfall_bps",
    ]

    for col in numeric_cols:
        a = merged[f"{col}_py"].to_numpy(dtype=float)
        b = merged[f"{col}_cpp"].to_numpy(dtype=float)
        diff = np.nanmax(np.abs(a - b))
        rows.append({
            "metric": col,
            "max_abs_diff": diff,
            "passed": bool(diff < 1e-8 or (np.isnan(diff) and np.all(np.isnan(a) == np.isnan(b)))),
            "note": "numeric parity",
        })

    event_cols = ["first_fill_event", "last_fill_event"]
    for col in event_cols:
        same = (merged[f"{col}_py"].to_numpy() == merged[f"{col}_cpp"].to_numpy()).all()
        rows.append({
            "metric": col,
            "max_abs_diff": 0.0 if same else 1.0,
            "passed": bool(same),
            "note": "integer parity",
        })

    same_reject = (merged["reject_reason_py"].to_numpy() == merged["reject_reason_cpp"].to_numpy()).all()
    rows.append({
        "metric": "reject_reason",
        "max_abs_diff": 0.0 if same_reject else 1.0,
        "passed": bool(same_reject),
        "note": "categorical parity",
    })

    same_complete = (merged["is_complete_py"].to_numpy() == merged["is_complete_cpp"].to_numpy()).all()
    rows.append({
        "metric": "is_complete",
        "max_abs_diff": 0.0 if same_complete else 1.0,
        "passed": bool(same_complete),
        "note": "boolean parity",
    })

    return pd.DataFrame(rows)

parity_report = compare_python_cpp(fills_python, fills_cpp)

parity_report

## 13. Benchmark timing

The C++ core should be faster for larger workloads.

This benchmark is simple:

- run Python reference engine;
- run C++ core if available;
- compare wall-clock time.

Do not over-interpret notebook timing. It depends on machine, compiler, and current load.

In [ ]:
def benchmark_engines(quotes, orders, config, lib_path):
    rows = []

    for repeat in range(config.benchmark_repeats):
        start = time.perf_counter()
        _ = execute_orders_python(quotes, orders, config)
        elapsed = time.perf_counter() - start
        rows.append({
            "engine": "python_reference",
            "repeat": repeat,
            "elapsed_seconds": elapsed,
            "orders_per_second": len(orders) / elapsed,
        })

    if lib_path is not None:
        # Warmup.
        _ = execute_orders_cpp(quotes, orders, config, lib_path)

        for repeat in range(config.benchmark_repeats):
            start = time.perf_counter()
            _ = execute_orders_cpp(quotes, orders, config, lib_path)
            elapsed = time.perf_counter() - start
            rows.append({
                "engine": "cpp_core",
                "repeat": repeat,
                "elapsed_seconds": elapsed,
                "orders_per_second": len(orders) / elapsed,
            })

    return pd.DataFrame(rows)

benchmark_report = benchmark_engines(quotes, orders, config, lib_path)

benchmark_summary = (
    benchmark_report
    .groupby("engine")
    .agg(
        mean_elapsed_seconds=("elapsed_seconds", "mean"),
        std_elapsed_seconds=("elapsed_seconds", "std"),
        mean_orders_per_second=("orders_per_second", "mean"),
    )
    .reset_index()
)

benchmark_report, benchmark_summary

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(benchmark_summary["engine"], benchmark_summary["mean_orders_per_second"])
plt.title("Engine Throughput")
plt.xlabel("Orders per second")
plt.ylabel("Engine")
plt.show()

## 14. Fill-quality diagnostics from selected engine

Use C++ results if available; otherwise use Python fallback.

In [ ]:
selected_engine = "cpp_core" if fills_cpp is not None and parity_report["passed"].all() else "python_reference"
selected_fills = fills_cpp if selected_engine == "cpp_core" else fills_python

selected_merged, selected_fill_summary, selected_fill_by_type = fill_quality_report(selected_fills, orders)

selected_engine, selected_fill_summary, selected_fill_by_type

## 15. Latency sensitivity

Latency can materially affect fill rates and slippage.

We rerun the engine across latency settings.

In [ ]:
latency_rows = []

for latency in [0, 1, 3, 5, 10, 20]:
    cfg = MicrostructureCppConfig(
        n_events=config.n_events,
        n_orders=config.n_orders,
        seed=config.seed,
        initial_mid=config.initial_mid,
        tick_size=config.tick_size,
        base_spread_ticks=config.base_spread_ticks,
        max_spread_ticks=config.max_spread_ticks,
        base_depth=config.base_depth,
        latency_events=latency,
        passive_fill_fraction=config.passive_fill_fraction,
        taker_fee_bps=config.taker_fee_bps,
        maker_rebate_bps=config.maker_rebate_bps,
        price_limit_pct=config.price_limit_pct,
        order_expiry_events=config.order_expiry_events,
        benchmark_repeats=1,
        output_subdir=config.output_subdir,
    )

    if lib_path is not None:
        fills = execute_orders_cpp(quotes, orders, cfg, lib_path)
    else:
        fills = execute_orders_python(quotes, orders, cfg)

    merged, summary, by_type = fill_quality_report(fills, orders)
    row = summary.iloc[0].to_dict()
    row["latency_events"] = latency
    latency_rows.append(row)

latency_sensitivity = pd.DataFrame(latency_rows)

latency_sensitivity

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(latency_sensitivity["latency_events"], latency_sensitivity["fill_rate_mean"], marker="o", label="mean fill rate")
plt.plot(latency_sensitivity["latency_events"], latency_sensitivity["completion_rate"], marker="o", label="completion rate")
plt.title("Fill Rate vs Latency")
plt.xlabel("Latency events")
plt.ylabel("Rate")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(latency_sensitivity["latency_events"], latency_sensitivity["mean_shortfall_bps"], marker="o")
plt.title("Mean Shortfall vs Latency")
plt.xlabel("Latency events")
plt.ylabel("Mean shortfall, bps")
plt.show()

## 16. Passive fill fraction sensitivity

The queue-position proxy is controlled by passive fill fraction.

Lower fill fraction means worse queue position.

In [ ]:
queue_rows = []

for passive_fill_fraction in [0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.00]:
    cfg = MicrostructureCppConfig(
        n_events=config.n_events,
        n_orders=config.n_orders,
        seed=config.seed,
        initial_mid=config.initial_mid,
        tick_size=config.tick_size,
        base_spread_ticks=config.base_spread_ticks,
        max_spread_ticks=config.max_spread_ticks,
        base_depth=config.base_depth,
        latency_events=config.latency_events,
        passive_fill_fraction=passive_fill_fraction,
        taker_fee_bps=config.taker_fee_bps,
        maker_rebate_bps=config.maker_rebate_bps,
        price_limit_pct=config.price_limit_pct,
        order_expiry_events=config.order_expiry_events,
        benchmark_repeats=1,
        output_subdir=config.output_subdir,
    )

    if lib_path is not None:
        fills = execute_orders_cpp(quotes, orders, cfg, lib_path)
    else:
        fills = execute_orders_python(quotes, orders, cfg)

    merged, summary, by_type = fill_quality_report(fills, orders)
    row = summary.iloc[0].to_dict()
    row["passive_fill_fraction"] = passive_fill_fraction
    queue_rows.append(row)

queue_sensitivity = pd.DataFrame(queue_rows)

queue_sensitivity

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(queue_sensitivity["passive_fill_fraction"], queue_sensitivity["fill_rate_mean"], marker="o", label="mean fill rate")
plt.plot(queue_sensitivity["passive_fill_fraction"], queue_sensitivity["completion_rate"], marker="o", label="completion rate")
plt.title("Fill Rate vs Passive Fill Fraction")
plt.xlabel("Passive fill fraction")
plt.ylabel("Rate")
plt.legend()
plt.show()

## 17. Stress-period diagnostics

We compare fills during normal and stressed quote regimes.

Orders submitted during stress should usually have:

- wider spreads;
- lower visible size;
- worse fill rates;
- worse slippage.

In [ ]:
order_stress = quotes.set_index("event_id").loc[orders["submit_event"], "stress"].to_numpy()
selected_merged["submit_stress"] = order_stress
selected_merged["submit_regime"] = np.where(selected_merged["submit_stress"] == 1, "stress", "normal")

stress_fill_report = (
    selected_merged
    .groupby("submit_regime")
    .agg(
        n_orders=("order_id", "count"),
        fill_rate_mean=("fill_rate", "mean"),
        completion_rate=("is_complete", "mean"),
        mean_shortfall_bps=("implementation_shortfall_bps", "mean"),
        p95_shortfall_bps=("implementation_shortfall_bps", lambda x: x.quantile(0.95)),
        total_notional=("notional", "sum"),
    )
    .reset_index()
)

stress_fill_report

## 18. Implementation shortfall decomposition

We decompose shortfall approximately into:

- spread cost proxy;
- fee cost;
- residual slippage.

This is a simplified TCA view.

In [ ]:
def approximate_shortfall_decomposition(selected_merged, quotes, orders, config):
    df = selected_merged.merge(
        orders[["order_id", "submit_event", "side", "arrival_mid"]],
        on="order_id",
        how="left",
        suffixes=("", "_ord"),
    )

    submit_spread = quotes.set_index("event_id").loc[df["submit_event"], "spread"].to_numpy()
    spread_bps_proxy = submit_spread / df["arrival_mid"].to_numpy() * 10000.0 / 2.0
    fee_bps = df["fees"] / df["notional"].replace(0, np.nan) * 10000.0

    df["spread_bps_proxy"] = spread_bps_proxy
    df["fee_bps"] = fee_bps
    df["residual_slippage_bps"] = df["implementation_shortfall_bps"] - df["spread_bps_proxy"] - df["fee_bps"]

    summary = pd.DataFrame([{
        "mean_shortfall_bps": df["implementation_shortfall_bps"].mean(),
        "mean_spread_bps_proxy": df["spread_bps_proxy"].mean(),
        "mean_fee_bps": df["fee_bps"].mean(),
        "mean_residual_slippage_bps": df["residual_slippage_bps"].mean(),
        "p95_shortfall_bps": df["implementation_shortfall_bps"].quantile(0.95),
    }])

    return df, summary

shortfall_decomp, shortfall_decomp_summary = approximate_shortfall_decomposition(selected_merged, quotes, orders, config)

shortfall_decomp_summary

## 19. Governance flags

A microstructure friction core should be reviewed if:

1. C++ is unavailable when required;
2. Python/C++ parity fails;
3. fill rates are unexpectedly low;
4. shortfall is too high;
5. latency sensitivity is severe;
6. queue sensitivity is severe;
7. stress fill quality deteriorates sharply;
8. benchmarks do not show speed improvement.

In [ ]:
cpp_available = fills_cpp is not None
parity_passed = bool(parity_report["passed"].all()) if cpp_available else False

fill_rate_mean = selected_fill_summary["fill_rate_mean"].iloc[0]
mean_shortfall = selected_fill_summary["mean_shortfall_bps"].iloc[0]
p95_shortfall = selected_fill_summary["p95_shortfall_bps"].iloc[0]

latency_fill_drop = (
    latency_sensitivity["fill_rate_mean"].iloc[0]
    - latency_sensitivity["fill_rate_mean"].iloc[-1]
)

queue_fill_range = (
    queue_sensitivity["fill_rate_mean"].max()
    - queue_sensitivity["fill_rate_mean"].min()
)

if {"normal", "stress"}.issubset(set(stress_fill_report["submit_regime"])):
    normal_shortfall = stress_fill_report.loc[stress_fill_report["submit_regime"] == "normal", "mean_shortfall_bps"].iloc[0]
    stress_shortfall = stress_fill_report.loc[stress_fill_report["submit_regime"] == "stress", "mean_shortfall_bps"].iloc[0]
    stress_shortfall_widening = stress_shortfall - normal_shortfall
else:
    stress_shortfall_widening = np.nan

if "cpp_core" in set(benchmark_summary["engine"]) and "python_reference" in set(benchmark_summary["engine"]):
    cpp_ops = benchmark_summary.loc[benchmark_summary["engine"] == "cpp_core", "mean_orders_per_second"].iloc[0]
    py_ops = benchmark_summary.loc[benchmark_summary["engine"] == "python_reference", "mean_orders_per_second"].iloc[0]
    speedup = cpp_ops / py_ops
else:
    speedup = np.nan

governance_flags = pd.DataFrame([{
    "selected_engine": selected_engine,
    "cpp_available": cpp_available,
    "parity_passed": parity_passed,
    "fill_rate_mean": fill_rate_mean,
    "mean_shortfall_bps": mean_shortfall,
    "p95_shortfall_bps": p95_shortfall,
    "latency_fill_drop": latency_fill_drop,
    "queue_fill_rate_range": queue_fill_range,
    "stress_shortfall_widening_bps": stress_shortfall_widening,
    "cpp_speedup_vs_python": speedup,
    "flag_cpp_unavailable": bool(not cpp_available),
    "flag_parity_failed": bool(cpp_available and not parity_passed),
    "flag_fill_rate_low": bool(fill_rate_mean < 0.50),
    "flag_mean_shortfall_high": bool(mean_shortfall > 15.0),
    "flag_p95_shortfall_high": bool(p95_shortfall > 50.0),
    "flag_latency_sensitivity_high": bool(latency_fill_drop > 0.20),
    "flag_queue_sensitivity_high": bool(queue_fill_range > 0.35),
    "flag_stress_widening_high": bool(stress_shortfall_widening > 10.0) if np.isfinite(stress_shortfall_widening) else False,
    "flag_cpp_speedup_low": bool(speedup < 2.0) if np.isfinite(speedup) else False,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_cpp_unavailable",
        "flag_parity_failed",
        "flag_fill_rate_low",
        "flag_mean_shortfall_high",
        "flag_p95_shortfall_high",
        "flag_latency_sensitivity_high",
        "flag_queue_sensitivity_high",
        "flag_stress_widening_high",
        "flag_cpp_speedup_low",
    ]
].any(axis=1)

governance_flags

## 20. Audit checks

Numerical and process checks:

1. bid is below ask;
2. prices align to tick;
3. orders have non-negative quantity;
4. fills never exceed order quantity;
5. remaining quantity is non-negative;
6. parity report exists;
7. selected engine has finite outputs;
8. governance flags exist.

In [ ]:
audit_rows = []

bid_ask_ok = bool((quotes["bid"] < quotes["ask"]).all())
audit_rows.append({
    "check": "bid_less_than_ask",
    "value": bid_ask_ok,
    "passed": bid_ask_ok,
})

tick_ok = bool(
    np.allclose(quotes["bid"] / config.tick_size, np.round(quotes["bid"] / config.tick_size))
    and np.allclose(quotes["ask"] / config.tick_size, np.round(quotes["ask"] / config.tick_size))
)
audit_rows.append({
    "check": "quote_prices_on_tick",
    "value": tick_ok,
    "passed": tick_ok,
})

qty_ok = bool((orders["quantity"] > 0).all())
audit_rows.append({
    "check": "order_quantities_positive",
    "value": qty_ok,
    "passed": qty_ok,
})

fills_not_exceed_qty = bool(
    (selected_merged["filled_qty"] <= selected_merged["quantity"] + 1e-8).all()
)
audit_rows.append({
    "check": "fills_do_not_exceed_order_quantity",
    "value": fills_not_exceed_qty,
    "passed": fills_not_exceed_qty,
})

remaining_nonnegative = bool((selected_merged["remaining_qty"] >= -1e-8).all())
audit_rows.append({
    "check": "remaining_quantity_nonnegative",
    "value": remaining_nonnegative,
    "passed": remaining_nonnegative,
})

parity_exists = bool(len(parity_report) > 0)
audit_rows.append({
    "check": "parity_report_exists",
    "value": parity_exists,
    "passed": parity_exists,
})

finite_selected = bool(
    np.isfinite(
        selected_fills[["filled_qty", "notional", "fees", "remaining_qty"]].to_numpy()
    ).all()
)
audit_rows.append({
    "check": "selected_fill_numeric_outputs_finite",
    "value": finite_selected,
    "passed": finite_selected,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 21. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

quotes.to_csv(output_dir / "l1_quotes.csv", index=False)
orders.to_csv(output_dir / "orders.csv", index=False)
fills_python.to_csv(output_dir / "fills_python.csv", index=False)
if fills_cpp is not None:
    fills_cpp.to_csv(output_dir / "fills_cpp.csv", index=False)

selected_fills.to_csv(output_dir / "fills_selected_engine.csv", index=False)
selected_merged.to_csv(output_dir / "fills_selected_merged.csv", index=False)
python_fill_summary.to_csv(output_dir / "python_fill_summary.csv", index=False)
python_fill_by_type.to_csv(output_dir / "python_fill_by_type.csv", index=False)
selected_fill_summary.to_csv(output_dir / "selected_fill_summary.csv", index=False)
selected_fill_by_type.to_csv(output_dir / "selected_fill_by_type.csv", index=False)
parity_report.to_csv(output_dir / "parity_report.csv", index=False)
benchmark_report.to_csv(output_dir / "benchmark_report.csv", index=False)
benchmark_summary.to_csv(output_dir / "benchmark_summary.csv", index=False)
latency_sensitivity.to_csv(output_dir / "latency_sensitivity.csv", index=False)
queue_sensitivity.to_csv(output_dir / "queue_sensitivity.csv", index=False)
stress_fill_report.to_csv(output_dir / "stress_fill_report.csv", index=False)
shortfall_decomp.to_csv(output_dir / "shortfall_decomposition.csv", index=False)
shortfall_decomp_summary.to_csv(output_dir / "shortfall_decomposition_summary.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "microstructure_friction_cpp_core_outputs",
    "schema_version": "microstructure_friction_cpp_core_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "cpp_path": str(cpp_path),
    "lib_path": str(lib_path) if lib_path is not None else None,
    "compile_message": compile_message,
    "selected_engine": selected_engine,
    "cpp_available": cpp_available,
    "parity_passed": parity_passed,
    "selected_fill_summary": selected_fill_summary.to_dict(orient="records"),
    "benchmark_summary": benchmark_summary.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook builds a Python reference execution engine and an optional C++ core.",
        "The execution model uses L1 bid/ask data only.",
        "Frictions include spread crossing, tick rounding, latency, L1 size caps, queue proxy, fees, rebates and price-limit rejection.",
        "C++ output is validated against the Python reference through parity tests.",
        "If no C++ compiler is available, the notebook falls back to Python-only mode.",
        "This is not a full exchange simulator; it is an auditable microstructure friction core."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "fills_selected_engine.csv", output_dir / "parity_report.csv", output_dir / "benchmark_summary.csv", output_dir / "governance_flags.csv", manifest_path

## 22. Practical checklist for a microstructure friction core

Before using a low-level fill core:

1. **Write a Python reference implementation.**
2. **Implement the C++ core only after rules are clear.**
3. **Keep memory layout explicit.**
4. **Test Python-vs-C++ parity.**
5. **Check tick rounding.**
6. **Check price-limit behaviour.**
7. **Stress latency assumptions.**
8. **Stress queue-position assumptions.**
9. **Benchmark throughput.**
10. **Document every simplification.**

## 23. Limitations

### L1 only

The model uses top-of-book bid/ask and visible size. It does not see deeper levels.

### Queue proxy

Passive fills use a simple fill fraction, not true queue position.

### No order persistence book

Limit orders are evaluated against future L1 snapshots but are not inserted into a simulated order book.

### No cancellations or replace logic

The model does not simulate cancel-replace priority loss.

### No hidden liquidity

Icebergs, dark pools and hidden orders are ignored.

### No matching-engine priority

Price-time priority is approximated, not simulated.

### C++ compiler optional

Notebook portability requires fallback to Python if compilation is unavailable.

## 24. Research frontier and extensions

Important extensions include:

1. full L2 order book replay;
2. queue-position modelling;
3. cancellation and replace priority;
4. hidden liquidity assumptions;
5. marketable-limit order edge cases;
6. exchange-specific price limits;
7. futures tick-size and contract multiplier support;
8. SIMD/vectorised C++ kernels;
9. Rust execution core;
10. production reconciliation against broker fills.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_07_latency_and_queue_position_model**  
   Replace passive fill fraction with queue dynamics.

2. **06_08_microprice_and_short_horizon_alpha**  
   Use L1 imbalance and microprice as short-horizon signals.

3. **06_09_execution_risk_controls_and_kill_switch**  
   Add safety controls around the C++ core.

4. **06_11_l1_bid_ask_backtest_execution_model**  
   Use L1 bid/ask fills in a full strategy backtest.

5. **06_12_production_reconciliation_and_breaks**  
   Reconcile simulated fills, paper fills and broker fills.

## 26. Summary

This notebook implemented a microstructure friction C++ core.

It showed how to:

1. simulate L1 bid/ask quote data;
2. generate market and limit orders;
3. model tick rounding;
4. model latency;
5. model market order fills;
6. model limit order fills;
7. cap fills by L1 visible size;
8. approximate queue position;
9. apply taker fees and maker rebates;
10. reject orders outside price limits;
11. compute implementation shortfall;
12. build a Python reference engine;
13. compile a C++ shared library;
14. wrap C++ with `ctypes`;
15. run Python-vs-C++ parity tests;
16. benchmark performance;
17. analyse latency and queue sensitivity;
18. analyse stress-period fill quality;
19. create governance flags and audit checks;
20. save outputs and manifests.

The key computational takeaway:

> A C++ execution core should be treated as a compiled hypothesis that must match a transparent Python reference.

The key financial takeaway:

> Microstructure frictions are not details; they are the difference between theoretical alpha and executable alpha.

## 27. Further reading

- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Abergel et al., *Limit Order Books.*
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*
- Bouchaud, Farmer and Lillo on market impact.
- Harris, *Trading and Exchanges.*
- Exchange technical documentation for tick sizes, matching rules and price limits.